# Motional Back-Action of the Stroboscopic Analysis Train

### What the measurement does *to* the oscillator — WP-W v0.6

*Strobo-Travels-Deep · `wp-wigner-tomography` · explanatory notebook · 2026-05-17*

This is a **readable explanatory notebook**, not a run report. It
builds up — in plain language first, equations second — what the
v0.6 motional back-action diagnostic means, then illustrates it with
a few small live computations and the committed result figure.

**Where this sits in WP-W.** The bulk of WP-W built and validated a
*reconstruction* chain: drive the ion with an analysis train, read
the spin, and recover the motional state's Wigner function
(deliverables P0/D2/D3) — then checked that chain against the real
simulation engine (D4). This notebook is the Rank-1 follow-up that
asks the **opposite, complementary question**: not *"what can we
reconstruct?"* but *"what did the act of measuring cost the
oscillator?"* — i.e. its **back-action**.

| Authoritative source | Where |
|---|---|
| Convention-locked derivation | [`notes/analytic_chain.md`](../wp-wigner-tomography/notes/analytic_chain.md) |
| Locked v0.6 scope | [`notes/back_action_scope.md`](../wp-wigner-tomography/notes/back_action_scope.md) |
| Work program | [`WORK-PROGRAM.md`](../wp-wigner-tomography/WORK-PROGRAM.md) §8 |
| Run record + audit | [`logbook/2026-05-16-back-action-run.md`](../wp-wigner-tomography/logbook/2026-05-16-back-action-run.md) |
| Committed data | `wp-wigner-tomography/numerics/back_action.h5` |

This notebook *uses* those results; it is **not** a source of truth.
Commits: `cb850a5` (scope) → `1b90f92` (run) → `8a2a8f9` (corrections).

## §1 Plain-language definitions

Read this once; every later section uses these words in exactly this
sense. No equations yet — those come in §2.

- **Analysis train.** The sequence of laser pulses that probes the
  ion's motion. Normally we run it, read the spin, and from the spin
  alone reconstruct the motion (the **χ readout** — "χ" is the
  characteristic function, the Fourier partner of the Wigner
  function). Back-action is the *reverse* viewpoint: forget the spin
  result, and ask what the train did to the motion itself.

- **Ideal SDF.** A *spin-dependent force*: a clean, instantaneous
  push on the oscillator whose direction depends on the ion's spin.
  It is the **reference model** — the perfect yardstick the real
  hardware is compared against. ("SDF" = state-dependent force.)

- **Branch separation.** The ideal SDF splits the motion into **two
  copies** — one for each spin direction. The copies move apart;
  the total distance between them is `β_tot`, and **each copy moves
  by half**, `±β_tot/2`. Each copy is a *branch*.

- **Unconditional vs conditional.** *Unconditional* = ignore the
  spin readout entirely (average over both spin outcomes) — this is
  what the oscillator looks like if you never look at the spin.
  *Conditional* = keep only the experimental runs where the spin
  came out a particular way.

- **Native engine.** The *actual* monochromatic Raman model — the
  real physics of the simulation/lab. It is **not expected** to
  reproduce the ideal SDF's clean two-branch split; understanding
  *how* it differs is the point.

- **Structural residual.** A difference between ideal and native
  that exists because the two have **genuinely different physical
  couplings** — not a knob you mis-set, and not something a
  calibration could remove. It is a property of the physics, so we
  *measure* it rather than tune it away.

- **Carrier tooth (k=0).** The train is run on the *carrier*
  resonance (no motional-sideband offset) — the simplest setting,
  and the one all earlier WP-W work used. (A sideband, k=1, is a
  documented future variant; see §6.)

## §2 The same ideas, precisely

Now the equations behind §1.

**The ideal SDF.** $\;U_\mathrm{SDF}(\beta_\mathrm{tot})
= D\!\big(\sigma_x\,\tfrac{\beta_\mathrm{tot}}{2}\big)$, where
$D(\xi)=e^{\xi a^\dagger-\xi^* a}$ is a phase-space displacement and
$\sigma_x$ is the spin operator that picks the branch. The
$|{+}x\rangle$ spin branch is displaced by $+\beta_\mathrm{tot}/2$,
$|{-}x\rangle$ by $-\beta_\mathrm{tot}/2$ → separation
$\beta_\mathrm{tot}$. (It must be $\sigma_x$, an *equatorial* axis,
not $\sigma_z$: only an equatorial axis rotates under the detuning
precession that builds the train's Dirichlet kernel — P1 confirmed
$10^{-14}$ with $\sigma_x$ vs $165\%$ with $\sigma_z$.)

**The post-train motional state.** Start the spin on the equator
($|{+}y\rangle$). *Unconditional* (spin ignored) gives a 50/50
mixture of the two displaced copies:

$$\rho_m^{(\mathrm{post})}
=\tfrac12\!\Big(D(\tfrac{\beta_\mathrm{tot}}{2})\,\rho_m\,D^\dagger(\tfrac{\beta_\mathrm{tot}}{2})
+D(-\tfrac{\beta_\mathrm{tot}}{2})\,\rho_m\,D^\dagger(-\tfrac{\beta_\mathrm{tot}}{2})\Big).$$

A spin **readout** instead post-selects one outcome; *which* basis
you read changes the conditioned state:

| read the spin in… | you keep | result |
|---|---|---|
| nothing (unconditional) | both outcomes | incoherent 50/50 mixture (decohered) |
| **σ_x** (the SDF axis) | one branch | a *single* displaced copy $D(\pm\beta_\mathrm{tot}/2)|\psi_m\rangle$ |
| **σ_y / σ_z** (equator) | a superposition | a *coherent* sum of the two copies — cat-like, with fringes |

**Numbers we report.** Purity $\mathrm{Tr}\,\rho^2$ (and *purity
drop* $1-\mathrm{Tr}\,\rho^2$ = how mixed the motion became);
fidelity $\mathcal F=\langle\psi_m^\mathrm{pre}|\rho_m^{(\mathrm{post})}|\psi_m^\mathrm{pre}\rangle$
(overlap with the state *before* the train); the Wigner function via
displaced parity $W(\alpha)=\tfrac{2}{\pi}\mathrm{Tr}[\rho\,D(\alpha)\Pi D^\dagger(\alpha)]$,
$\Pi=(-1)^{a^\dagger a}$ (the $2/\pi$ fixes $W_\mathrm{vac}(0)=2/\pi$);
and the **structural residual** = the $L^1$ area between the ideal
and native unconditional Wigner functions when both are driven by
the *identical* control program.

> **Convention trap (worth stating).** This engine is **not**
> textbook-signed: `apply_mw_pi2(|↓⟩,0)` is
> $|{+}y\rangle=(|{\downarrow}\rangle-i|{\uparrow}\rangle)/\sqrt2$
> and $\sigma_y=-2\,\mathrm{Im}\langle d|u\rangle$. A σ_y readout
> written with generic signs shipped *flipped* and an expert audit
> missed it — caught only by a smoke test that a known eigenstate
> post-selects its own σ at probability 1. Anchor σ conventions to
> the engine's numerical locks, never to remembered algebra.

## §3 Setup

In [ ]:
import os, sys
import numpy as np
import h5py
import matplotlib.pyplot as plt

REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(REPO, "scripts"))
sys.path.insert(0, os.path.join(REPO, "wp-wigner-tomography", "numerics"))

from stroboscopic import HilbertSpace, states
from stroboscopic.ideal_sdf import build_ideal_sdf_train

# Reuse the committed, smoke-tested helpers verbatim (8/8 locks) so this
# notebook cannot drift from the validated pipeline.
from _common import (
    partial_trace_spin, conditional_motional_ket, wigner_from_rho,
    cat_ket, inverse_dirichlet, W_mixed_cat, W_coherent, _displacement,
)

plt.rcParams.update({"figure.dpi": 110, "image.cmap": "RdBu_r"})

OMEGA_M, BETA0, N, K = 1.3, 0.05, 30, 0          # carrier tooth k=0
H5 = os.path.join(REPO, "wp-wigner-tomography", "numerics", "back_action.h5")
print("setup OK — production: "
      f"N={N}, beta0={BETA0}, k={K} (carrier), back_action.h5 "
      f"{'present' if os.path.exists(H5) else 'MISSING'}")

## §4 First, does the machinery tell the truth? (the vacuum check)

Before trusting *any* back-action number we check the one case with
a known exact answer. Start from the **vacuum** (the oscillator at
rest), apply the ideal SDF, ignore the spin. The result must be the
50/50 mixture of two displaced vacua — for which purity, fidelity,
and the whole Wigner function have **textbook closed forms**:

$$\mathrm{Tr}\,\rho^2=\tfrac12\big(1+e^{-|\beta_\mathrm{tot}|^2}\big),
\quad
\mathcal F=e^{-|\beta_\mathrm{tot}|^2/4},
\quad
W=W_{\mathrm{mixed\,cat}}(\beta_\mathrm{tot}/2).$$

This is the back-action analogue of WP-W's P0/P1 gates: if the live
code reproduces these to machine precision, the parity-form Wigner,
the partial trace, and the spin conventions are all correct. **What
to look for below:** the residuals printed should be ~$10^{-14}$, and
the "measured" and "analytic" Wigner panels should be visually
identical (their difference flat).

In [ ]:
def ideal_post_state(psi_m, beta_tot_abs, theta=0.0, nmax=40):
    # |↓⟩ψ_m → MW π/2 (|+y⟩) → ideal-SDF train; return ψ_out.
    # Peak probe ⇒ ratio=N ⇒ x=0 (on-resonance carrier, φ_train=0).
    ratio = beta_tot_abs / BETA0
    x = inverse_dirichlet(N, ratio)
    phi = (theta - (N - 1) * x / 2.0) % (2.0 * np.pi)
    delta = K * OMEGA_M + x / (2.0 * np.pi / OMEGA_M)
    hs = HilbertSpace(n_spins=1, mode_cutoffs=(nmax,))
    psi0 = np.concatenate([psi_m, np.zeros(nmax, dtype=np.complex128)])
    psi_eq = states.apply_mw_pi2(psi0, mw_phase_deg=0.0, nmax=nmax)
    train = build_ideal_sdf_train(hs=hs, beta0=BETA0, ac_phase_rad=phi,
                                  omega_m=OMEGA_M, delta=delta,
                                  n_pulses=N, k_sideband=K)
    return train.evolve(psi_eq)

NMAX = 40
beta_tot = N * BETA0                       # "peak" probe: 1.5 (real, θ=0)
psi_m0 = states.fock_state(0, NMAX)
psi_out = ideal_post_state(psi_m0, beta_tot, nmax=NMAX)
rho = partial_trace_spin(psi_out, NMAX)

purity = float(np.real(np.trace(rho @ rho)))
fid = float(np.real(np.vdot(psi_m0, rho @ psi_m0)))
p_tgt = 0.5 * (1.0 + np.exp(-beta_tot**2))
f_tgt = np.exp(-beta_tot**2 / 4.0)
print(f"|beta_tot| = {beta_tot:.3f}  (each branch moves +-{beta_tot/2:.3f})")
print(f"purity   {purity:.12f}  vs 1/2(1+e^-|b|^2) = {p_tgt:.12f}   |d|={abs(purity-p_tgt):.1e}")
print(f"fidelity {fid:.12f}  vs e^-|b|^2/4      = {f_tgt:.12f}   |d|={abs(fid-f_tgt):.1e}")

In [ ]:
ax = np.linspace(-3, 3, 31)
AX, AY = np.meshgrid(ax, ax)
W_meas, werr = wigner_from_rho(rho, ax)
W_tgt = W_mixed_cat(AX + 1j*AY, (beta_tot/2.0) + 0j)
print(f"max|W_measured - W_analytic| = {np.max(np.abs(W_meas-W_tgt)):.1e}"
      f"   (here NMAX={NMAX}; the production run at NMAX=60 reaches 1.3e-10)")
print(f"max|Im W| = {werr:.0e}   -> Wigner stays real: no orientation/sign bug")

fig, axs = plt.subplots(1, 3, figsize=(12, 3.6), constrained_layout=True)
v = np.max(np.abs(W_meas))
for a, dat, t in zip(axs, [W_meas, W_tgt, W_meas - W_tgt],
        ["measured  W(rho_post)", "exact prediction", "difference (should be flat)"]):
    im = a.imshow(dat, origin="lower", extent=(ax[0],ax[-1],ax[0],ax[-1]),
                  vmin=-v, vmax=v); a.set_title(t, fontsize=10)
    a.set_xlabel("Re alpha"); fig.colorbar(im, ax=a, shrink=0.8)
axs[0].set_ylabel("Im alpha")
fig.suptitle("Vacuum check: live code vs the exact answer (the only hard gate)")
plt.show()

**Takeaway.** The residuals are at the floating-point floor and
the difference panel is flat: the back-action machinery is exact.
Every number after this point can be trusted. (The Wigner residual
here is slightly larger only because this live cell uses a smaller
oscillator basis than the production run — a basis-size artefact, not
an error.)

## §5 One push, three ways of looking — the readout dependence

Now the conceptual heart. Take that *same* post-train wavefunction
(vacuum, ideal SDF) and look at it three ways. **What to look for:**
the three panels are the *same physics*, differing only in what we
do with the spin — yet they look completely different.

In [ ]:
ket_xp, p_xp = conditional_motional_ket(psi_out, NMAX, "x", +1)
ket_yp, p_yp = conditional_motional_ket(psi_out, NMAX, "y", +1)
Wx, _ = wigner_from_rho(np.outer(ket_xp, np.conj(ket_xp)), ax)
Wy, _ = wigner_from_rho(np.outer(ket_yp, np.conj(ket_yp)), ax)

tgt = _displacement(beta_tot/2.0, NMAX) @ psi_m0   # analytic single branch
print(f"sigma_x=+1 : prob={p_xp:.3f}, overlap with one displaced branch "
      f"D(+b/2)|0> = {abs(np.vdot(tgt,ket_xp))**2:.6f}  (exact single branch)")
print(f"sigma_y=+1 : prob={p_yp:.3f}  (engine |+y> convention; smoke-locked)")

fig, axs = plt.subplots(1, 3, figsize=(12, 3.6), constrained_layout=True)
for a, dat, t in zip(axs, [W_meas, Wx, Wy],
        ["ignore the spin", "keep sigma_x = +1", "keep sigma_y = +1"]):
    v = np.max(np.abs(dat))
    im = a.imshow(dat, origin="lower", extent=(ax[0],ax[-1],ax[0],ax[-1]),
                  vmin=-v, vmax=v); a.set_title(t, fontsize=11)
    a.set_xlabel("Re alpha"); fig.colorbar(im, ax=a, shrink=0.8)
axs[0].set_ylabel("Im alpha")
fig.suptitle("One post-train wavefunction, three spin readouts (vacuum)")
plt.show()

**Reading the three panels:**

- **Ignore the spin** → *two displaced blobs, no fringes.* The
  oscillator split into two copies and, because we threw the spin
  information away, they add up incoherently — a classical-looking
  mixture. This is the back-action "cost": the motion is now mixed.
- **Keep σ_x = +1** → *one blob; no cat.* Reading the SDF axis
  *selects a single branch* — a clean displaced vacuum (overlap with
  $D(+\beta_\mathrm{tot}/2)|0\rangle$ is exactly 1).
- **Keep σ_y = +1** → *coherent branch sum; fringes appear.* Reading
  the equator keeps both copies in superposition — a Schrödinger-cat
  with quantum interference (the faint blue/red ripples).

Same wavefunction, three stories. Which one you get is a *choice of
what to measure*, not different physics.

## §6 Ideal vs the real drive — the structural residual

The ideal SDF is a fiction; the lab runs the **native engine** (the
monochromatic Raman model). We drive *both* with the **identical
control program** — same detunings, same phases, same train length,
same input, on the carrier (k=0). Any difference left is therefore
**structural**: it comes from the two having different physical
couplings, not from a mis-set knob. Quantifying that difference is a
*third* independent ideal-vs-real disagreement number for WP-W
(alongside the two from the D4 bridge).

We load the committed `back_action.h5` so the figures show the exact,
provenance-bound production numbers (not a re-run). **What to look
for:** the ideal column splits cleanly into branches; the native
column does *not*, and the "σ_x-branch" overlap collapses from 1 to
well below 1 — the structural residual made quantitative.

In [ ]:
inputs = ["vacuum", "fock2", "cat1.5"]
LBL = {"vacuum": "vacuum", "fock2": "Fock |2>", "cat1.5": "cat |a|=1.5"}
with h5py.File(H5, "r") as f:
    axh = f["alpha_axis"][:]
    Wpre = {i: f[f"W_pre/{i}"][:] for i in inputs}
    recs = []
    for k in f:
        if not k.startswith("rec_"): continue
        g = f[k]
        d = {a: g.attrs[a] for a in g.attrs}
        d["W"] = g["W"][:]
        d["cond_W_sy_plus"] = g["cond_W_sy_plus"][:] if "cond_W_sy_plus" in g else None
        d["sx_branch_fid"] = float(g["cond/x+"].attrs["branch_fidelity"])
        recs.append(d)
    gate = [dict(f[f"gate/{p}"].attrs) for p in f["gate"]]

print("Vacuum gate (committed run):")
for r in gate:
    print(f"  {r['point']:>4}: purity d={r['purity_resid']:.0e}  "
          f"fidelity d={r['fidelity_resid']:.0e}  W d={r['W_max_resid']:.0e}  "
          f"{'PASS' if r['pass'] else 'FAIL'}")
print("\nHow branch-like is each (peak probe)?  1.0 = perfect clean split")
print(f"  {'input':>10} {'ideal':>7} {'native':>8}   {'ideal-vs-native W area':>22}")
for inp in inputs:
    pk = {r["leg"]: r for r in recs if r["input"]==inp and r["point"]=="peak"}
    print(f"  {inp:>10} {pk['ideal']['sx_branch_fid']:>7.3f} "
          f"{pk['native']['sx_branch_fid']:>8.3f}   "
          f"{float(pk['ideal']['ideal_vs_native_W_L1']):>22.3f}")

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(14, 10.5), constrained_layout=True)
cols = ["before the train", "after: ideal SDF", "after: native engine",
        "after: ideal, keep sigma_y=+1"]
for ri, inp in enumerate(inputs):
    pk = {r["leg"]: r for r in recs if r["input"]==inp and r["point"]=="peak"}
    panels = [Wpre[inp], pk["ideal"]["W"], pk["native"]["W"],
              pk["ideal"]["cond_W_sy_plus"]]
    v = max(np.max(np.abs(p)) for p in panels)
    for ci, (W, ct) in enumerate(zip(panels, cols)):
        a = axes[ri, ci]
        im = a.imshow(W, origin="lower", extent=(axh[0],axh[-1],axh[0],axh[-1]),
                      vmin=-v, vmax=v)
        if ri == 0: a.set_title(ct, fontsize=10)
        if ci == 0: a.set_ylabel(LBL[inp] + "\nIm alpha", fontsize=10)
        a.set_xlabel("Re alpha", fontsize=8)
        fig.colorbar(im, ax=a, shrink=0.8)
    for ci, leg in ((1,"ideal"), (2,"native")):
        r = pk[leg]
        axes[ri,ci].text(0.03,0.97,
            f"mixedness {float(r['purity_drop']):.2f}\nkept {float(r['fidelity_to_pre']):.2f}",
            transform=axes[ri,ci].transAxes, va="top", fontsize=8,
            bbox=dict(boxstyle="round", fc="white", alpha=0.7))
fig.suptitle("Ideal clean displacement vs the real Raman drive "
             "(same control program; committed back_action.h5)", fontsize=12)
plt.show()

**Reading the grid (rows = input state; peak probe):**

- **after: ideal SDF** — the input splits into two clean copies
  exactly as the reference model predicts.
- **after: native engine** — *the carrier native drive disturbs the
  oscillator differently, and much less like a clean branch split.*
  At the carrier it mostly rotates the spin and barely displaces the
  motion (vacuum stays almost unchanged), or it shears the state
  (the cat row) instead of splitting it. The "how branch-like"
  table makes this quantitative: ideal = 1.000 always; native
  collapses to 0.04–0.35.
- **after: ideal, keep σ_y=+1** — the cat-like coherent sum (fringes)
  from §5, now for each input.

The single number per row labelled *ideal-vs-native W area* is the
**structural residual**: large (0.8–1.9) precisely because the real
monochromatic engine is not, and cannot be tuned into, the ideal
SDF — a physics statement, not a calibration gap.

## §7 What we currently understand

1. **The machinery is exact** — the vacuum check reproduces the
   closed forms to ~$10^{-14}$ (state metrics) and the parity-form
   Wigner to $1.3\times10^{-10}$ in production. Trustworthy.

2. **The ideal SDF cleanly splits the motion into two branches**, and
   *what you measure on the spin decides what you get*: ignore it →
   a decohered mixture (the measurement cost); read the SDF axis →
   one branch; read the equator → a coherent cat. One wavefunction,
   three stories.

3. **The real engine does not reproduce that split — and we now have
   a number for how far off it is.** At matched control on the
   carrier the native "branch-likeness" collapses from 1 to
   0.04–0.35 and the ideal-vs-native Wigner area is $O(1)$. This is
   the §7#3 *structural* gap turned into a quantitative,
   reproducible residual — the third such number for WP-W.

4. **Honest limits.** Carrier only (k=0): the native engine's richer
   sideband back-action (collapse–revival) is a documented next
   variant, not done here. Wigner residuals carry a basis-size floor,
   always quoted with that caveat. Deferred and decision-gated: the
   k=1 sideband, the full thermal input set, and squeezed-vacuum.

## §8 References

Verified subset (full bibliography + per-paper extractions live in
[`WORK-PROGRAM.md`](../wp-wigner-tomography/WORK-PROGRAM.md)
§References — kept there deliberately, not duplicated here).

- **[CG69]** Cahill & Glauber, *Density Operators and Quasiprobability
  Distributions*, Phys. Rev. **177**, 1882 (1969).
  [doi:10.1103/PhysRev.177.1882](https://doi.org/10.1103/PhysRev.177.1882)
  — phase-space formalism; the Wigner ↔ characteristic-function pair.
- **[LD97]** Lutterbach & Davidovich, Phys. Rev. Lett. **78**, 2547
  (1997). [doi:10.1103/PhysRevLett.78.2547](https://doi.org/10.1103/PhysRevLett.78.2547)
  — the displaced-parity route used to compute every Wigner here.
- **[FH20]** Flühmann & Home, Phys. Rev. Lett. **125**, 043602
  (2020). [doi:10.1103/PhysRevLett.125.043602](https://doi.org/10.1103/PhysRevLett.125.043602)
  — the σ_x SDF + direct χ readout WP-W inherits.
- **[Hasse24]** Hasse, Palani, Thomm, Warring, Schaetz, Phys. Rev. A
  **109**, 053105 (2024).
  [doi:10.1103/PhysRevA.109.053105](https://doi.org/10.1103/PhysRevA.109.053105)
  — the stroboscopic protocol WP-W reinterprets; App. D maps the
  back-action qualitatively (WP-W adds the quantitative ideal-vs-native
  difference).
- **[Clos16]** Clos, Porras, Warring, Schaetz, Phys. Rev. Lett.
  **117**, 170401 (2016).
  [doi:10.1103/PhysRevLett.117.170401](https://doi.org/10.1103/PhysRevLett.117.170401)
  — Hasse-group predecessor establishing the interaction form.
- **[LBMW03]** Leibfried, Blatt, Monroe, Wineland, Rev. Mod. Phys.
  **75**, 281 (2003).
  [doi:10.1103/RevModPhys.75.281](https://doi.org/10.1103/RevModPhys.75.281)
  — trapped-ion Lamb–Dicke / spin–motion reference.